# Retail E-commerce Sales Analytics Pipeline
## Phase 1: Setup + Bronze Layer (Raw Ingestion)

This notebook covers:
- Catalog/Schema/Volume setup
- Loading all batch CSVs (orders, customers, products, stores) into Bronze Delta tables
- Loading all incremental/CDC CSVs into their own Bronze tables
- Adding audit metadata columns: `source_file`, `ingestion_ts`, `load_type`

## 0. Setup — Catalog, Schemas, Volume

### Upload the project folder
Upload the entire `retail_delta_project` folder (with its `datasets/batch` and
`datasets/incremental` subfolders intact) into the volume created above —
via **Catalog → retail_demo → raw → retail_files → Upload**, or by using
`dbutils.fs.cp` after uploading a zip and unzipping it.

After uploading, update `BASE_PATH` below to point at wherever the
`retail_delta_project` folder actually landed.

In [0]:
CATALOG ="retail_demo"  
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# UPDATE THIS to match where you uploaded the retail_delta_project folder
BASE_PATH =  "/Volumes/workspace/raw/retail_files/retail_delta_project"
BATCH_PATH = f"{BASE_PATH}/datasets/batch"
INCR_PATH = f"{BASE_PATH}/datasets/incremental"
CHECKPOINT_PATH = f"{BASE_PATH}/_checkpoints"
SCHEMA_PATH = f"{BASE_PATH}/_schemas"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {RAW_SCHEMA}")

print("BASE_PATH:", BASE_PATH)
print("BATCH_PATH:", BATCH_PATH)
print("INCR_PATH:", INCR_PATH)

BASE_PATH: /Volumes/workspace/raw/retail_files/retail_delta_project
BATCH_PATH: /Volumes/workspace/raw/retail_files/retail_delta_project/datasets/batch
INCR_PATH: /Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental


In [0]:
dbutils.fs.rm(f"{CHECKPOINT_PATH}/orders_incremental", recurse=True)
dbutils.fs.rm(f"{CHECKPOINT_PATH}/customers_cdc", recurse=True)
dbutils.fs.rm(f"{CHECKPOINT_PATH}/products_cdc", recurse=True)
dbutils.fs.rm(f"{SCHEMA_PATH}/orders_incremental", recurse=True)
dbutils.fs.rm(f"{SCHEMA_PATH}/customers_cdc", recurse=True)
dbutils.fs.rm(f"{SCHEMA_PATH}/products_cdc", recurse=True)
print("Checkpoints and schema locations cleared.")

Checkpoints and schema locations cleared.


In [0]:
# Quick sanity check: list what's actually in BASE_PATH so we catch path mistakes early
display(dbutils.fs.ls(BASE_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/raw/retail_files/retail_delta_project/CEI_Intern_Project_Guide_Celebal.pdf,CEI_Intern_Project_Guide_Celebal.pdf,407975,1783318686000
dbfs:/Volumes/workspace/raw/retail_files/retail_delta_project/PROJECT_GUIDE_updated.md,PROJECT_GUIDE_updated.md,13450,1783318684000
dbfs:/Volumes/workspace/raw/retail_files/retail_delta_project/_checkpoints/,_checkpoints/,0,1783783053418
dbfs:/Volumes/workspace/raw/retail_files/retail_delta_project/_schemas/,_schemas/,0,1783783053418
dbfs:/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/,datasets/,0,1783783053418


## 1. Bronze Layer — Batch Files
We read everything as **strings** (no schema enforcement, no date parsing) —
this is intentional: Bronze should never fail because of a malformed row.
All cleaning/casting happens later, in Silver.

In [0]:
from pyspark.sql import functions as F

def read_csv_as_strings(path):
    """Read a CSV with every column as a string — safe, flexible Bronze ingestion."""
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)   # keep everything as string
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
            .csv(path))

def add_audit_columns(df, load_type):
    """Add the required Bronze audit metadata columns."""
    return (df
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("load_type", F.lit(load_type)))

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail_demo;
CREATE SCHEMA IF NOT EXISTS retail_demo.raw;
CREATE SCHEMA IF NOT EXISTS retail_demo.silver;
CREATE SCHEMA IF NOT EXISTS retail_demo.gold;
CREATE VOLUME IF NOT EXISTS retail_demo.raw.retail_files;

In [0]:
# --- orders_batch ---
orders_batch_df = add_audit_columns(
    read_csv_as_strings(f"{BATCH_PATH}/orders_batch.csv"), "batch"
)
(orders_batch_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders"))
print("bronze_orders:", spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders").count())

bronze_orders: 12180


In [0]:
# --- customers_batch ---
customers_batch_df = add_audit_columns(
    read_csv_as_strings(f"{BATCH_PATH}/customers_batch.csv"), "batch"
)
(customers_batch_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers"))
print("bronze_customers:", spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers").count())

bronze_customers: 2560


In [0]:
# --- products_batch ---
products_batch_df = add_audit_columns(
    read_csv_as_strings(f"{BATCH_PATH}/products_batch.csv"), "batch"
)
(products_batch_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{RAW_SCHEMA}.bronze_products"))
print("bronze_products:", spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_products").count())

bronze_products: 830


In [0]:
# --- stores_batch ---
stores_batch_df = add_audit_columns(
    read_csv_as_strings(f"{BATCH_PATH}/stores_batch.csv"), "batch"
)
(stores_batch_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{RAW_SCHEMA}.bronze_stores"))
print("bronze_stores:", spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_stores").count())

bronze_stores: 80


## 2. Bronze Layer — Incremental / CDC Files
These live in per-day folders (`datasets/incremental/day_YYYY-MM-DD/`). We use
**Auto Loader** (`cloudFiles`) so that if new day-folders are added later, we
don't have to manually track which files were already processed — Auto Loader
keeps its own state in the checkpoint location.

Note: day `2026-04-26`'s orders file introduces a new column (`coupon_code`).
Auto Loader's schema evolution handles this automatically (`mergeSchema`).

In [0]:
# --- orders_incremental (Auto Loader, with schema evolution for the Day-3 coupon_code column) ---
orders_incr_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/orders_incremental")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .load(f"{INCR_PATH}/*/orders_incremental_*.csv")
)

orders_incr_stream = add_audit_columns(orders_incr_stream, "incremental")

(orders_incr_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/orders_incremental")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders_incremental")
    .awaitTermination())

print("bronze_orders_incremental:", spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders_incremental").count())

bronze_orders_incremental: 6135


In [0]:
# --- customers_cdc (Auto Loader) ---
customers_cdc_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/customers_cdc")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .load(f"{INCR_PATH}/*/customers_cdc_*.csv")
)

customers_cdc_stream = add_audit_columns(customers_cdc_stream, "incremental")

(customers_cdc_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/customers_cdc")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_cdc")
    .awaitTermination())

print("bronze_customers_cdc:", spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_cdc").count())

bronze_customers_cdc: 630


In [0]:
# --- products_cdc (Auto Loader) ---
products_cdc_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/products_cdc")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .load(f"{INCR_PATH}/*/products_cdc_*.csv")
)

products_cdc_stream = add_audit_columns(products_cdc_stream, "incremental")

(products_cdc_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/products_cdc")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{RAW_SCHEMA}.bronze_products_cdc")
    .awaitTermination())

print("bronze_products_cdc:", spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_products_cdc").count())

bronze_products_cdc: 180


## 3. Verification — All 7 Bronze Tables

In [0]:
bronze_tables = [
    "bronze_orders", "bronze_orders_incremental",
    "bronze_customers", "bronze_customers_cdc",
    "bronze_products", "bronze_products_cdc",
    "bronze_stores",
]

for t in bronze_tables:
    full_name = f"{CATALOG}.{RAW_SCHEMA}.{t}"
    cnt = spark.table(full_name).count()
    print(f"{t:35s} -> {cnt} rows")

bronze_orders                       -> 12180 rows
bronze_orders_incremental           -> 6135 rows
bronze_customers                    -> 2560 rows
bronze_customers_cdc                -> 630 rows
bronze_products                     -> 830 rows
bronze_products_cdc                 -> 180 rows
bronze_stores                       -> 80 rows


In [0]:
# Peek at the incremental orders table — confirm coupon_code appeared for day 2026-04-26 only
display(
    spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders_incremental")
    .filter(F.col("coupon_code").isNotNull())
    .limit(5)
)

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,ingest_date,coupon_code,_rescued_data,source_file,ingestion_ts,load_type
OI3000003,2026-04-23 00:00:00,C01378,P00014,S045,6,24907.63,0.1,141973.49,CARD,shipped,2026-04-26,NEW10,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-11T15:17:54.626Z,incremental
OI3000004,2026-04-26 14:17:00,C02220,P00747,S030,2,61937.51,0.05,117681.27,CARD,shipped,2026-04-26,NEW10,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-11T15:17:54.626Z,incremental
OI3000017,2026-04-26 15:58:00,C02376,P00391,S020,2,40752.6,0.2,73354.68,WALLET,returned,2026-04-26,FESTIVE20,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-11T15:17:54.626Z,incremental
OI3000018,2026-04-26 23:13:00,C01463,P00329,S033,6,4843.15,0.0,26153.01,WALLET,delivered,2026-04-26,FESTIVE20,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-11T15:17:54.626Z,incremental
OI3000019,2026-04-26 05:01:00,C00141,P00334,S057,6,16032.79,0.0,91386.9,COD,returned,2026-04-26,FESTIVE20,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-11T15:17:54.626Z,incremental


In [0]:
history_df = spark.sql("DESCRIBE HISTORY retail_demo.raw.bronze_orders_incremental")
display(history_df.select("version", "timestamp", "operation", "operationMetrics"))

version,timestamp,operation,operationMetrics
2,2026-07-11T15:17:57.000Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 6135, numOutputBytes -> 113271, numAddedFiles -> 1)"
1,2026-07-11T14:44:48.000Z,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0, numAddedFiles -> 0)"
0,2026-07-11T14:44:40.000Z,CREATE TABLE,Map()


## Summary — Phase 1 (Bronze Layer)
- Created the `retail_demo` catalog with `raw`, `silver`, `gold` schemas and a
  Unity Catalog volume to hold the project files.
- Loaded all 4 historical batch files (orders, customers, products, stores)
  into Bronze Delta tables, reading every column as a string to avoid failures
  on dirty data.
- Loaded all 3 incremental/CDC file types across all day-folders using
  **Auto Loader** (`cloudFiles`), which automatically tracks which files have
  already been processed and evolves the schema automatically — this is how
  the Day-3 `coupon_code` column was picked up without any manual schema work.
- Added the required audit columns (`source_file`, `ingestion_ts`, `load_type`)
  to every Bronze table.
- Verified row counts across all 7 Bronze tables.